# Energy Sector Financial Analysis

This notebook integrates the data fetcher, WACC calculator, and DCF model to determine intrinsic value.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add project root to path so we can import from src
sys.path.append(os.path.abspath('..'))

from src.data_fetchers import DataFetcher
from src.models.wacc import WACCCalculator
from src.models.dcf import DCFModel
from config.settings import COMPANIES, SECTOR_ASSUMPTIONS, OIL_SCENARIOS, COMPANY_FINANCIALS, GLOBAL_DEFAULTS

# Styling for charts
plt.style.use('ggplot')
print("✅ Environment Initialized")

In [ ]:
# Initialize Fetcher
fetcher = DataFetcher()

# Set Ticker (Change this to analyze different companies)
ticker = "XOM"
print(f"📡 Fetching data for {ticker}...")

# 1. Get Stock Price
price_data = fetcher.get_stock_price(ticker)
current_price = fetcher.get_current_price(ticker)
print(f"   Current Price: ${current_price:.2f}")

# 2. Get Financials from Database with Safety Net
print(f"📚 Loading financial data for {ticker}...")
if ticker in COMPANY_FINANCIALS:
    financials = COMPANY_FINANCIALS[ticker]
else:
    print(f"⚠️ Warning: Using Global Defaults for {ticker}")
    financials = {
        "revenue": 100e9, 
        "net_debt": 10e9, 
        "shares": 1e9
    }

latest_revenue = financials["revenue"]
net_debt = financials["net_debt"]
shares_out = financials["shares"]

print(f"   Base Revenue: ${latest_revenue/1e9:.1f}B")
print(f"   Net Debt:     ${net_debt/1e9:.1f}B")
print(f"   Shares Out:   {shares_out/1e9:.1f}B")

In [ ]:
# Calculate WACC
print(f"🧮 Calculating Risk Premium for {ticker}...")
wacc_calc = WACCCalculator(ticker)
wacc_result = wacc_calc.calculate_wacc()

print(f"   Beta: {wacc_result['beta']}")
print(f"   Cost of Equity: {wacc_result['cost_of_equity']:.2%}")
print(f"   WACC: {wacc_result['wacc']:.2%}")

In [ ]:
# Load Sector Assumptions
sector = COMPANIES.get(ticker, {'type': 'integrated'})['type']
assumptions = SECTOR_ASSUMPTIONS[sector]

# Initialize DCF Model
dcf = DCFModel(
    ticker=ticker,
    wacc=wacc_result['wacc'],
    terminal_growth_rate=assumptions['terminal_growth']
)

current_rev = latest_revenue
projected_fcf = []

print(f"\n📉 Projecting Cash Flows ({sector.upper()} Sector Logic):")
for i in range(5):
    growth_rate = assumptions['revenue_growth'][i]
    current_rev = current_rev * (1 + growth_rate)
    
    fcf = dcf.calculate_fcf(
        revenue=current_rev,
        ebitda_margin=assumptions['ebitda_margin_base'],
        tax_rate=assumptions['tax_rate'],
        capex_percent_revenue=assumptions['capex_percent_revenue']
    )
    projected_fcf.append(fcf)

# Calculate Terminal Value
tv = dcf.calculate_terminal_value(projected_fcf[-1])
valuation = dcf.discount_cash_flows(projected_fcf, tv)
print(f"   Enterprise Value: ${valuation['enterprise_value']/1e9:,.1f}B")

In [ ]:
# Convert Enterprise Value to Equity Value
equity_value = valuation['enterprise_value'] - net_debt

# LOGIC GUARD: Distressed Asset Check
if equity_value < 0:
    print(f"⚠️ DISTRESSED SIGNAL: Debt (${net_debt/1e9:.1f}B) exceeds Enterprise Value.")
    print("   Equity value floored at $0.00.")
    implied_share_price = 0.00
    upside = -1.0 
else:
    implied_share_price = equity_value / shares_out
    upside = (implied_share_price - current_price) / current_price

print(f"\n⚖️ VALUATION VERDICT for {ticker}")
print(f"   Current Market Price:   ${current_price:.2f}")
print(f"   DCF Implied Value:      ${implied_share_price:.2f}")
print(f"   Upside / (Downside):    {upside:.1%}")

In [ ]:
from src.models.sensitivity import SensitivityAnalysis

# 1. Run the Sensitivity Logic
print("\n🌪️ RUNNING STRESS TEST...")
sens = SensitivityAnalysis(ticker, wacc_result['wacc'], assumptions['terminal_growth'])
sens_results = sens.run_oil_price_sensitivity(
    base_revenue=latest_revenue, 
    tax_rate=assumptions['tax_rate'], 
    capex_percent=assumptions['capex_percent_revenue'],
    net_debt=net_debt,
    shares=shares_out
)

# 2. Visualize the "Football Field" Chart
scenarios = list(sens_results.keys())
prices = list(sens_results.values())
colors = ['firebrick', 'steelblue', 'forestgreen'] # Bear, Base, Bull

plt.figure(figsize=(10, 6))
bars = plt.bar(scenarios, [max(0, p) for p in prices], color=colors, alpha=0.8)

# Add a line for Current Market Price
plt.axhline(y=current_price, color='black', linestyle='--', linewidth=2, label=f'Current Market Price (${current_price:.2f})')

# Add labels
name = COMPANIES.get(ticker, {'name': ticker})['name']
plt.title(f'{name} ({ticker}) Valuation Sensitivity', fontsize=14)
plt.ylabel('Implied Share Price ($)', fontsize=12)
plt.xlabel('Oil Price Scenario', fontsize=12)
plt.legend()

# Add value labels on top of bars
for i, bar in enumerate(bars):
    height = prices[i] # Original value
    label = f'${height:.2f}' if height >= 0 else 'Distressed'
    plt.text(bar.get_x() + bar.get_width()/2., max(0, height),
             label,
             ha='center', va='bottom', fontsize=12, fontweight='bold', color='red' if height < 0 else 'black')

plt.tight_layout()
plt.show()